In [ ]:
# INSTALL & IMPORTS
!pip install -q sentence-transformers rank_bm25 Sastrawi pandas numpy scikit-learn

import os
import sys
import re
import string
import numpy as np
import pandas as pd
import torch
from google.colab import drive

from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

print("[INFO] Library installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.8 MB/s eta 0:00:00


[INFO] Library installed and imported.


In [ ]:
# CONNECT DRIVE & PATH CONFIG
drive.mount('/content/drive')

# Konfigurasi Path
BASE_PATH = "/content/drive/MyDrive/FP_Quran_Project"
DATA_PATH = os.path.join(BASE_PATH, "data")
MODEL_PATH = os.path.join(BASE_PATH, "models")

# Cek Folder
if os.path.exists(DATA_PATH):
    print(f"[SUCCESS] Folder Data ditemukan: {DATA_PATH}")
else:
    print(f"[ERROR] Folder tidak ditemukan: {DATA_PATH}")
    print("Pastikan struktur folder di Google Drive sudah benar!")

# Set Device (GPU/CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Running on: {device.upper()}")

Mounted at /content/drive
[SUCCESS] Folder Data ditemukan: /content/drive/MyDrive/FP_Quran_Project/data
[INFO] Running on: CPU


In [ ]:
# LOAD DATASET
csv_file = os.path.join(DATA_PATH, "quran_tafsir_clean.csv")
df_tafsir = pd.read_csv(csv_file)

# Normalisasi Nama Kolom (Agar tidak error key)
possible_cols = {'tafsir', 'text', 'isi', 'content'}
found_col = set(df_tafsir.columns).intersection(possible_cols)

if 'tafsir_text' in df_tafsir.columns:
    pass
elif found_col:
    old_col = list(found_col)[0]
    df_tafsir.rename(columns={old_col: 'tafsir_text'}, inplace=True)
    print(f"[INFO] Kolom '{old_col}' diubah menjadi 'tafsir_text'.")
else:
    # Fallback: Ambil kolom object terakhir (biasanya teks panjang)
    obj_cols = df_tafsir.select_dtypes(include=['object']).columns
    df_tafsir.rename(columns={obj_cols[-1]: 'tafsir_text'}, inplace=True)
    print(f"[WARN] Auto-detect: Menggunakan kolom '{obj_cols[-1]}' sebagai teks.")

# Ambil List Corpus
corpus_texts = df_tafsir['tafsir_text'].tolist()
print(f"[INFO] {len(corpus_texts)} ayat berhasil dimuat.")

# Mapping Index untuk Evaluasi Nanti
lookup_map = {}
for idx, row in df_tafsir.iterrows():
    lookup_map[(row['surah'], row['ayah'])] = idx

[INFO] Kolom 'tafsir' diubah menjadi 'tafsir_text'.
[INFO] 6236 ayat berhasil dimuat.


In [ ]:
# LOAD SBERT MODEL
try:
    sbert_path = os.path.join(MODEL_PATH, "sbert_finetuned_quran")
    model_retriever = SentenceTransformer(sbert_path, device=device)
    print("   [OK] Menggunakan Model Fine-Tuned (Local).")
except:
    print("   [WARN] Model Fine-Tuned tidak ketemu/error. Menggunakan Base MPNet.")
    model_retriever = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2', device=device)

The tokenizer you are loading from '/content/drive/MyDrive/FP_Quran_Project/models/sbert_finetuned_quran' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


   [OK] Menggunakan Model Fine-Tuned (Local).


In [ ]:
# BUILD INDEX (DENSE & SPARSE)

# Dense Index (Embeddings)
CACHE_FILE = os.path.join(DATA_PATH, "cache_sbert_corpus.pt")

if os.path.exists(CACHE_FILE):
    emb_check = torch.load(CACHE_FILE, map_location='cpu')
    if emb_check.shape[1] == model_retriever.get_sentence_embedding_dimension():
        corpus_embeddings = emb_check.to(device)
        print("[CACHE] Embeddings dimuat dari file.")
    else:
        print("[INFO] Dimensi cache beda. Menghitung ulang embeddings...")
        corpus_embeddings = model_retriever.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)
        torch.save(corpus_embeddings, CACHE_FILE)
else:
    print("[INFO] Menghitung Embeddings baru...")
    corpus_embeddings = model_retriever.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)
    torch.save(corpus_embeddings, CACHE_FILE)

# Sparse Index (BM25)
print("[INFO] Membangun Index BM25...")
tokenized_corpus = [doc.lower().split(" ") for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)
print("[SUCCESS] Indexing Selesai.")

[CACHE] Embeddings dimuat dari file.
[INFO] Membangun Index BM25...
[SUCCESS] Indexing Selesai.


In [ ]:
# DEFINE HYBRID SEARCH FUNCTION
def hybrid_search(query, top_k=50, alpha=0.5):
    """
    alpha: Bobot penyeimbang.
    - alpha = 1.0 -> 100% SBERT (Semantic)
    - alpha = 0.0 -> 100% BM25 (Keyword)
    - alpha = 0.5 -> Balanced
    """
    # SBERT SCORE (DENSE)
    q_vec = model_retriever.encode(query, convert_to_tensor=True)
    dense_scores = util.cos_sim(q_vec, corpus_embeddings)[0].cpu().numpy()

    # BM25 SCORE (SPARSE)
    tokenized_query = query.lower().split(" ")
    sparse_scores = bm25.get_scores(tokenized_query)

    # NORMALISASI (Min-Max Scaling) agar range skor seimbang
    def normalize(scores):
        if scores.max() == scores.min(): return scores
        return (scores - scores.min()) / (scores.max() - scores.min())

    dense_norm = normalize(dense_scores)
    sparse_norm = normalize(sparse_scores)

    # FUSION (Weighted Sum)
    final_scores = (alpha * dense_norm) + ((1 - alpha) * sparse_norm)

    # SORTING & TOP-K
    top_indices = np.argpartition(-final_scores, range(top_k))[:top_k]
    sorted_top_indices = sorted(top_indices, key=lambda i: final_scores[i], reverse=True)

    return sorted_top_indices

In [ ]:
# DATASET UJI (GROUND TRUTH)
semantic_test_set = {
    # LEVEL 1: LOW-LEVEL (Keyword Specific)
    "Low-Level": [
        {"q": "Hukum memakan daging babi", "truth": [("Al-Mā'idah", 3), ("Al-Baqarah", 173)]},
        {"q": "Ayat tentang pembagian warisan anak", "truth": [("An-Nisā'", 11)]},
        {"q": "Perintah mendirikan shalat", "truth": [("Al-Baqarah", 43), ("Hūd", 114), ("Al-Isrā'", 78)]},
        {"q": "Larangan mendekati zina", "truth": [("Al-Isrā'", 32)]},
        {"q": "Kewajiban puasa ramadhan", "truth": [("Al-Baqarah", 183), ("Al-Baqarah", 185)]},
        {"q": "Larangan riba dan bunga bank", "truth": [("Al-Baqarah", 275), ("Al-Baqarah", 278), ("Āli ‘Imrān", 130)]},
        {"q": "Hukum meminum khamar dan berjudi", "truth": [("Al-Mā'idah", 90), ("Al-Baqarah", 219)]},
        {"q": "Perintah berbakti kepada orang tua (birrul walidain)", "truth": [("Al-Isrā'", 23), ("Luqmān", 14)]},
        {"q": "Kewajiban ibadah haji bagi yang mampu", "truth": [("Āli ‘Imrān", 97), ("Al-Baqarah", 196)]},
        {"q": "Tentang lailatul qadar malam kemuliaan", "truth": [("Al-Qadr", 1), ("Al-Qadr", 2), ("Al-Qadr", 3)]}
    ],

    # LEVEL 2: MID-LEVEL (Deskriptif / Kisah)
    "Mid-Level": [
        {"q": "Ciri-ciri orang munafik yang suka berdusta", "truth": [("Al-Munāfiqūn", 1), ("Al-Baqarah", 8), ("An-Nisā'", 142)]},
        {"q": "Balasan neraka bagi pemakan harta anak yatim", "truth": [("An-Nisā'", 10)]},
        {"q": "Kisah Nabi Musa membelah lautan dengan tongkat", "truth": [("Asy-Syu‘arā'", 63), ("Ṭāhā", 77)]},
        {"q": "Kondisi manusia yang panik di hari kiamat", "truth": [("‘Abasa", 34), ("‘Abasa", 35), ("‘Abasa", 36), ("‘Abasa", 37), ("Al-Qāri‘ah", 4)]},
        {"q": "Etika sopan santun saat bertamu ke rumah orang", "truth": [("An-Nūr", 27), ("An-Nūr", 28)]},
        {"q": "Kisah pemuda yang tertidur di dalam gua (ashabul kahfi)", "truth": [("Al-Kahf", 9), ("Al-Kahf", 10), ("Al-Kahf", 11)]},
        {"q": "Nasihat Luqman kepada anaknya agar tidak syirik", "truth": [("Luqmān", 13)]},
        {"q": "Makanan penghuni neraka pohon zaqqum", "truth": [("Aṣ-Ṣāffāt", 62), ("Ad-Dukhān", 43), ("Al-Wāqi‘ah", 52)]},
        {"q": "Doa nabi yunus saat berada dalam perut ikan", "truth": [("Al-Anbiyā'", 87)]},
        {"q": "Wanita yang menggoda nabi yusuf", "truth": [("Yūsuf", 23), ("Yūsuf", 30)]}
    ],

    # LEVEL 3: HIGH-LEVEL (Abstrak / Implisit / Filosofis)
    "High-Level": [
        {"q": "Bagaimana Islam memandang toleransi antar agama", "truth": [("Al-Kāfirūn", 6), ("Al-Baqarah", 256), ("Al-Mumtaḥanah", 8)]},
        {"q": "Cara menjaga ketenangan jiwa saat tertimpa musibah", "truth": [("Al-Baqarah", 155), ("Al-Baqarah", 156), ("Ar-Ra‘d", 28)]},
        {"q": "Kedudukan wanita setara dengan pria dalam amal shaleh", "truth": [("An-Nisā'", 124), ("An-Naḥl", 97), ("Al-Aḥzāb", 35)]},
        {"q": "Tujuan eksistensial penciptaan manusia dan jin", "truth": [("Aż-Żāriyāt", 56)]},
        {"q": "Pentingnya persatuan dan larangan berpecah belah", "truth": [("Āli ‘Imrān", 103), ("Al-Anfāl", 46)]},
        {"q": "Sikap rendah hati saat berjalan di muka bumi", "truth": [("Al-Furqān", 63), ("Al-Isrā'", 37)]},
        {"q": "Larangan berbuat kerusakan di muka bumi (lingkungan)", "truth": [("Ar-Rūm", 41), ("Al-A‘rāf", 56)]},
        {"q": "Pentingnya musyawarah dalam mengambil keputusan", "truth": [("Āli ‘Imrān", 159), ("Asy-Syūrā", 38)]},
        {"q": "Etos kerja keras dan tidak bermalas-malasan", "truth": [("At-Taubah", 105), ("Al-Jumu‘ah", 10)]},
        {"q": "Konsep tolong menolong dalam kebaikan bukan kejahatan", "truth": [("Al-Mā'idah", 2)]}
    ]
}

print(f"[INFO] Dataset Uji Diperbarui: Total {sum(len(v) for v in semantic_test_set.values())} Query.")

[INFO] Dataset Uji Diperbarui: Total 30 Query.


In [ ]:
# METRICS CALCULATION HELPER
def get_ground_truth_indices(truth_list):
    indices = []
    for t in truth_list:
        if t in lookup_map:
            indices.append(lookup_map[t])
    return indices

def calculate_metrics(retrieved_indices, true_indices, k=10):
    retrieved_k = retrieved_indices[:k]

    # Recall
    intersection = set(retrieved_k) & set(true_indices)
    recall = len(intersection) / len(true_indices) if true_indices else 0

    # MRR
    mrr = 0
    for i, idx in enumerate(retrieved_k):
        if idx in true_indices:
            mrr = 1 / (i + 1)
            break

    # nDCG
    relevance = [1 if idx in true_indices else 0 for idx in retrieved_k]
    if len(relevance) < k: relevance += [0] * (k - len(relevance))

    ideal = [1] * len(true_indices) + [0] * (k - len(true_indices))
    ideal = ideal[:k]

    try:
        ndcg = ndcg_score([ideal], [relevance], k=k)
    except:
        ndcg = 0
    return mrr, recall, ndcg

In [ ]:
# RUN EVALUATION
final_report = []
for level, items in semantic_test_set.items():
    level_mrr, level_recall, level_ndcg = [], [], []

    for item in items:
        query = item['q']
        true_indices = get_ground_truth_indices(item['truth'])

        if not true_indices: continue

        # RUN HYBRID SEARCH 
        results = hybrid_search(query, top_k=10, alpha=0.5)

        # CALC METRICS
        mrr, recall, ndcg = calculate_metrics(results, true_indices)
        level_mrr.append(mrr)
        level_recall.append(recall)
        level_ndcg.append(ndcg)

    # Average per Category
    final_report.append({
        "Kategori": level,
        "MRR": np.mean(level_mrr),
        "Recall@10": np.mean(level_recall),
        "nDCG@10": np.mean(level_ndcg)
    })

# Tampilkan Tabel
df_res = pd.DataFrame(final_report)
print("\nHASIL AKHIR:")
print(df_res.round(4).to_markdown(index=False))